In [18]:
import pandas as pd
import re
import json

In [19]:
df = pd.read_csv("../KC_PET_ACP_CTLSTT_LC_DATA_2023.csv")

In [20]:
import pymysql

def map_pet_only(text):
    if text is None or str(text).strip() == "":
        return 0

    text = str(text)

    if text == "반려동물 전용":
        return 1

    # "해당없음" 포함 + 기타 값 전부 0 처리
    return 0

# DB 연결
conn = pymysql.connect(
    host="localhost",
    user="mini",
    password="mini",
    db="miniproject",
    charset="utf8mb4"
)
cursor = conn.cursor()

# place 테이블에서 id, fclty_nm, ADDR 가져오기
cursor.execute("SELECT id, fclty_nm, LNM_ADDR FROM place")
place_map = {
    (str(name).strip(), str(addr).strip()): pid
    for pid, name, addr in cursor.fetchall()
}

# insert SQL
sql = """
INSERT INTO infofilterdata (
    place_id,
    pet_info,
    PET_INFO_CN
) VALUES (%s, %s, %s)
"""

# df iterate
for _, r in df.iterrows():
    place_name = str(r["FCLTY_NM"]).strip()
    place_addr = str(r["LNM_ADDR"]).strip()  # 주소 컬럼
    oper_text = r["PET_INFO_CN"]

    place_id = place_map.get((place_name, place_addr))
    if place_id is None:
        print(f"❗ place_id 매핑 실패: {place_name} / {place_addr}")
        continue

    pet_info = map_pet_only(oper_text)

    cursor.execute(sql, (place_id, pet_info, oper_text))

conn.commit()
conn.close()